In [52]:
import sys
import numpy as np
import pandas as pd
from copy import deepcopy
from SciExpeM_API.SciExpeM import SciExpeM
my_sciexpem = SciExpeM(username='YOUR_USERNAME', password='YOUR_PASSWORD')
# SciExpeM(username='YOUR_USERNAME', password='YOUR_PASSWORD', secure=False, verify=False, warning=False)
# database: porta 5432 all'indirizzo 127.0.0.1
from SciExpeM_API.Models import *

CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}

Attention. SciExpeM is a singleton.


In [49]:
ID_ChemModels = [606, 618, 619, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633]
labels = ['NATPROT','2ndO2Add','Abs','betaC','betaH','Class5','ConcEl','CycEth','init','initH','ketoDec','ketoForm','LT_Iso','O2Add','QOOHDec','QOOHHO2']
exps = [3867, 3866, 3865, 3864, 3856]
conditions = ['55bar_909-1115K_phi1', '20bar_806-1048K_phi1', 
              '10bar_1236-1344K_phi1', '2bar_1249-1378K_phi1',
               '28bar_751-1127K_phi1', '43bar_692-1264K_phi1',]

In [67]:
ID_ChemModels = [606, 618, 619, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633]
labels = ['NATPROT','2ndO2Add','Abs','betaC','betaH','Class5','ConcEl','CycEth','init','initH','ketoDec','ketoForm','LT_Iso','O2Add','QOOHDec','QOOHHO2']
exps = [3856]
conditions = ['28bar_751-1127K_phi1c',]

In [68]:

for jj, j in enumerate(exps):
    exp = my_sciexpem.filterDatabase(model_name='Experiment', id = j)[0].experimental_data[0]
    exp['temperature [K]'] = 1000/exp['temperature [K]'].values
    exp = pd.DataFrame(exp[['temperature [K]', 'ignition delay [us]']].values, 
                       columns = ['1000/T [K]', 'IDT exp [us]' ])
    allexp = deepcopy(exp)
    for i, ID_ChemModel in enumerate(ID_ChemModels):
        exec = my_sciexpem.filterDatabase(model_name='Execution', chemModel = ID_ChemModel, experiment = j)[0]
        x = 1000/exec.simulation_results[0]['ParametricAnalysisIDT']['T0'].values
        y = np.array(exec.simulation_results[0]['ParametricAnalysisIDT']['tau_P(slope)'].values)* 1e6
        if len(x) == len(exp['1000/T [K]']):
            sim = pd.DataFrame(np.array([y]).T * 1e6, columns=[ 'IDT [mus] ' + labels[i]])
        else:
            sim = pd.DataFrame(np.array([x, y]).T , columns=['1000/T [K]' , 'IDT [mus] ' + labels[i]])
        allexp = pd.concat([allexp,sim], axis=1)
    print(allexp)        
    with pd.ExcelWriter('IDTs.xlsx', mode='a') as writer:
        allexp.to_excel(writer, sheet_name=conditions[jj])


    1000/T [K]  IDT exp [us]  1000/T [K]  IDT [mus] NATPROT  1000/T [K]  \
0     1.331558        1165.0    1.333333        1269.836941    1.333333   
1     1.322751         958.0    1.250000         625.923835    1.250000   
2     1.310616        1087.0    1.176471         599.037688    1.176471   
3     1.250000         829.0    1.111111         757.033371    1.111111   
4     1.221001         729.0    1.052632         876.631330    1.052632   
5     1.199041         806.0    1.000000         665.574988    1.000000   
6     1.152074         844.0    0.952381         462.692135    0.952381   
7     1.131222        1014.0    0.909091         341.011100    0.909091   
8     1.123596        1018.0    0.869565         258.679404    0.869565   
9     1.121076         981.0         NaN                NaN         NaN   
10    1.117318        1144.0         NaN                NaN         NaN   
11    1.086957        1216.0         NaN                NaN         NaN   
12    1.082251        136

In [65]:
sim

,IDT [mus] QOOHHO2
0,1260.299069
1,588.241271
2,501.629481
3,554.901616
4,577.132446
5,454.213656
6,342.932201
7,274.449743
8,223.331947
